In [23]:
import pandas as pd
import glob
import numpy as np
import matplotlib as plt
import seaborn as sns

file_paths = glob.glob('cleaned_data.csv')

df_list = []

for file in file_paths:
    temp_df = pd.read_csv(file,encoding= 'utf-8')
    df_list.append(temp_df)

df = pd.concat(df_list,ignore_index = True)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794627 entries, 0 to 794626
Data columns (total 17 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   접수연도      794627 non-null  int64  
 1   자치구명      794627 non-null  object 
 2   법정동명      794627 non-null  object 
 3   지번구분      794627 non-null  int64  
 4   지번구분명     741440 non-null  object 
 5   건물명       794627 non-null  object 
 6   계약일       794627 non-null  object 
 7   물건금액(억원)  794627 non-null  float64
 8   건물면적(㎡)   794627 non-null  float64
 9   토지면적(㎡)   418845 non-null  float64
 10  층         794627 non-null  int64  
 11  건축년도      794627 non-null  int64  
 12  건물용도      794627 non-null  object 
 13  평수        794627 non-null  float64
 14  평당단가(억원)  794627 non-null  float64
 15  계약연도      794627 non-null  int64  
 16  계약월       794627 non-null  int64  
dtypes: float64(5), int64(6), object(6)
memory usage: 103.1+ MB


In [24]:
dong_trade = (
    df.groupby(["자치구명","법정동명","건물용도","계약연도"])
    .size()
    .reset_index(name="거래건수")
)

dong_trade


,자치구명,법정동명,건물용도,계약연도,거래건수
0,강남구,개포동,단독다가구,2018,9
1,강남구,개포동,단독다가구,2019,8
2,강남구,개포동,단독다가구,2020,14
3,강남구,개포동,단독다가구,2021,10
4,강남구,개포동,단독다가구,2022,9
...,...,...,...,...,...
8260,중랑구,중화동,오피스텔,2020,9
8261,중랑구,중화동,오피스텔,2021,4
8262,중랑구,중화동,오피스텔,2022,2
8263,중랑구,중화동,오피스텔,2023,7


# “동” TOP10

In [122]:
target_year = 2024
target_gu = "성동구"

dong_by_gu = (
    dong_trade[(dong_trade["계약연도"] == target_year)&
    (dong_trade["자치구명"] == target_gu)]
    .sort_values("거래건수", ascending=False)
)

In [123]:
dong_top10 = (
    dong_by_gu.groupby("법정동명",as_index=False)["거래건수"].sum()
        .sort_values("거래건수", ascending=False)
    )

dong_top10.head(5)

,법정동명,거래건수
14,하왕십리동,437
15,행당동,425
11,옥수동,347
8,성수동1가,234
3,금호동4가,187


# “건물용도” TOP10

In [124]:
use_top10 = (
    dong_by_gu.sort_values("거래건수", ascending = False)
    .loc[:, ["법정동명", "건물용도", "거래건수"]]
    .head(10)
)

use_top10

,법정동명,건물용도,거래건수
3755,하왕십리동,아파트,410
3782,행당동,아파트,388
3680,옥수동,아파트,337
3489,금호동4가,아파트,180
3605,성수동1가,아파트,163
3425,금호동1가,아파트,143
3734,응봉동,아파트,142
3447,금호동2가,아파트,138
3542,마장동,아파트,100
3468,금호동3가,아파트,99


In [125]:
dong_main_use = (
    dong_by_gu.sort_values(["법정동명", "거래건수"], ascending=[True, False])
              .groupby("법정동명", as_index=False)
              .head(1)
              .loc[:, ["법정동명", "건물용도", "거래건수"]]
              .sort_values("거래건수", ascending=False)
              
)
dong_main_use.head(20)


,법정동명,건물용도,거래건수
3755,하왕십리동,아파트,410
3782,행당동,아파트,388
3680,옥수동,아파트,337
3489,금호동4가,아파트,180
3605,성수동1가,아파트,163
3425,금호동1가,아파트,143
3734,응봉동,아파트,142
3447,금호동2가,아파트,138
3542,마장동,아파트,100
3468,금호동3가,아파트,99


In [126]:
pivot = (
    dong_by_gu.pivot_table(index="법정동명", # 행을 동으로 만듦, 동 당 한 줄
                           columns="건물용도", # 열을 건물 용도로 만듦
                           values="거래건수", # 표 안에 들어갈 값
                           aggfunc="sum", # 같은 동, 용도 조합이 여러 행이면 합계로 집계
                           fill_value=0) # 해당 동에서 어떤 용도가 거래가 없으면 NaN 대신 0으로 채움
)

# 동별 총거래 기준 내림차순 정렬
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
# pivot.sum(axis=1): 행방향 합계
# .sort_values(ascending=False): 총 거래건수 큰 순으로 정렬
# .index: 그 정렬된 "동 이름 순서"만 뽑음
# pivot.loc: pivot을 그 순서대로 재배치

pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100
pivot_pct = pivot_pct.round(1)
pivot_pct

건물용도,단독다가구,아파트,연립다세대,오피스텔
법정동명,,,,
하왕십리동,1.6,93.8,0.7,3.9
행당동,0.9,91.3,5.9,1.9
옥수동,0.0,97.1,0.3,2.6
성수동1가,5.1,69.7,14.1,11.1
금호동4가,0.5,96.3,3.2,0.0
응봉동,0.0,78.0,22.0,0.0
성수동2가,9.4,40.0,33.9,16.7
금호동1가,0.6,90.5,8.9,0.0
금호동2가,2.0,91.4,6.6,0.0
